# Search MSD Entries in the Dictionaries

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

root_dir = Path.home() / 'scratch/projects/befunddolmetscher'

In [ ]:
import sys

sys.path.append((root_dir / 'search').as_posix())

### load MSD data

In [ ]:
import json

with (root_dir / 'data/msd_20251024.json').open('r') as f:
    msd = json.load(f)
msd = {d['instance_id']: d for d in msd}
len(msd)

In [ ]:
from sqlitevec import SQLiteVec

msd_table = {}
for lang in ['en', 'de']:
    msd_table[lang] = SQLiteVec(
        table=f"msd_doc_{lang}",
        db_file="msd.db",
        lang=lang,
    )

### load lookup-tables

Download `*.db` from HeiBox and put it in the same directory.

In [ ]:
from helpers import load_dictionary_tables, load_data_tables

icd10_table, mesh_table, do_table, pschyrembel_table = load_dictionary_tables()

In [ ]:
# placeholder to store the ids
external_ids = {}
for instance_id, r in msd.items():
    tmp = {k: set([]) for k in set(list(r['external_ids'].keys()) + ['ICD-10', 'ICD-10CM', 'MeSH', 'DOID', 'Pschyrembel'])}
    for k, v in r['external_ids'].items():
        if isinstance(v, list):
            for u in v:
                if isinstance(u, dict):
                    tmp[k].add(u['value'])
    external_ids[instance_id] = tmp

In [ ]:
external_ids['9-35']

### lookup MSD entries in the dictionaries

This is a simple word-based lookup. It will match, i.e. "lymph" to all dictionary entries of lymph diseases. We will validate it later using similarity.

In [ ]:
from helpers import lookup_title, _add_to_dict
from tqdm.auto import tqdm

for instance_id, r in tqdm(msd.items()):
    queries = {'en': set([]), 'de': set([])}
    for lang in ['en', 'de']:
        for ed, d in r[lang].items():
            queries[lang].add(d['topic']['name'])
            if 'alternative_name' in d['topic'] and isinstance(d['topic']['alternative_name'], list):
                for name in d['topic']['alternative_name']:
                    queries[lang].add(name)

    for keys, dict_table in [
        (['icd10'], icd10_table),
        (['descriptor_ui'], mesh_table),
        (['doid', 'icd10', 'mesh'], do_table),
        (['pschyrembel_id', 'icd10'], pschyrembel_table),
    ]:
        for lang in ['en', 'de']:
            if lang in dict_table:
                for query in queries[lang]:
                    hits = lookup_title(table=dict_table[lang], query=query, keys=keys)
                    external_ids = _add_to_dict(hits, external_ids, instance_id)


In [ ]:
external_ids['9-35']

### assign found external_ids with distance

it will take a while

In [ ]:
from helpers import search_by_similarity, check_by_similarity
from tqdm.auto import tqdm

for instance_id, r in tqdm(msd.items()):
    emb = {}
    for lang in ['en', 'de']:
        # we use the embedding from amateur articles,
        # because we want to make the distance comparable
        # with similarity to other data instances (gesund.bund, apotheken umschau, etc.)
        msd_found = msd_table[lang].metadata_search(instance_id=instance_id, audience="amateur")
        emb[lang] = msd_found[0]['embedding']

    for name, key, dict_table in [
        ('ICD-10CM', 'icd10', icd10_table),
        ('MeSH', 'descriptor_ui', mesh_table),
        ('DOID', 'doid', do_table),
        ('Pschyrembel', 'pschyrembel_id', pschyrembel_table),
    ]:
        if name not in r['external_ids'] or r['external_ids'][name] is None:
            if len(external_ids[instance_id][name]) > 0:
                ids = list(external_ids[instance_id][name])
                distances = check_by_similarity(dict_table, emb, ids, key, limit=25)
            else:
                distances = search_by_similarity(msd_table, dict_table, emb, instance_id, key, limit=5)
            r['external_ids'][name] = [{'value': k, 'distance': v} for k, v in distances.items()] if len(distances) > 0 else None


In [ ]:
msd['9-35']['external_ids']

In [ ]:
with (root_dir / 'data/msd_20251029.json').open('w') as f:
    json.dump([r for _, r in msd.items()], f, indent=4)

### add texts to the database

In [ ]:
import json

vectors = []
for output_path in (root_dir / 'results').glob('msd_embed_*.jsonl'):
    vectors.extend([json.loads(l) for l in output_path.read_text().splitlines()])

vectors = {(v['instance_id'], v['language'], v['edition'], v['paragraph_id']): v['embedding'][0] for v in vectors if len(v['embedding'][0]) == 4096}
len(vectors) #

In [ ]:
from tqdm.auto import tqdm

titles = {'en': [], 'de': []}
texts = {'en': [], 'de': []}
metadatas = {'en': [], 'de': []}
embeds = {'en': [], 'de': []}

for instance_id, r in tqdm(msd.items()):
    for lang in ['en', 'de']:
        for ed, d in r[lang].items():
            para = {p['type']: p for p in d['paragraphs']}
            if '_summary' in para and '_first-paragraph' in para:
                paragraph_id = '_summary_first-paragraph'
            else:
                paragraph_id = d['paragraphs'][0]['paragraph_id']

            titles[lang].append(d['topic']['name'])
            texts[lang].append(d['plain_text'])
            metadatas[lang].append({'instance_id': instance_id, 'language': lang, 'audience': ed})
            embeds[lang].append(vectors[(instance_id, lang, ed, paragraph_id)])

In [ ]:
from sqlitevec import SQLiteVec

msd_table = {}
for lang in ['de', 'en']:
    msd_table[lang] = SQLiteVec.from_texts(
        titles=titles[lang],
        texts=texts[lang],
        embeds=embeds[lang],
        metadatas=metadatas[lang],
        table=f"msd_doc_{lang}",
        db_file="msd.db",
        lang=lang,
    )